# SSL Watermarking in Google Colab

This notebook is set up for a folder-based workflow: it fits PCA-whitening on many natural images, generates keys, then watermarks an entire image folder instead of a single sample image.

Use a folder of training/reference images for PCA fitting, then point the batch-marking cells at any folder you want to watermark.


## 1) Install dependencies

Run this once per Colab session.

In [ ]:
!pip -q install -r requirements.txt

## 2) Mount Drive and locate the repo

If you uploaded the repository directly into `/content/ssl`, this cell will use it. Otherwise it tries Google Drive.

In [ ]:
from pathlib import Path
import os

from google.colab import drive

drive.mount('/content/drive')

candidate_roots = [
    Path('/content/ssl'),
    Path('/content/drive/MyDrive/ssl'),
    Path('/content/drive/MyDrive/thesis/ssl'),
]
REPO_DIR = next((path for path in candidate_roots if (path / 'src').exists()), None)
if REPO_DIR is None:
    raise FileNotFoundError(
        'Could not find the repo. Put the project folder in /content/ssl or Google Drive, then rerun this cell.'
    )

os.chdir(REPO_DIR)
print(f'Using repo: {REPO_DIR}')
print('Files:', sorted(p.name for p in REPO_DIR.iterdir())[:20])


## 5) Fit PCA-whitening on many images

Use a folder of natural images. This is the closest step in this repo to "training": the whitening transform is fit from a large set of reference images before watermarking starts.


In [ ]:
!python scripts/fit_pca.py --images-dir "{TRAIN_DIR}" --out "{WHITENING_PATH}" --max-images 2000 --batch-size 16 --image-size 224


## 6) Generate watermark keys

Zero-bit uses one carrier vector. Multi-bit uses `k` orthonormal carriers.


In [ ]:
from pathlib import Path

TRAIN_DIR = Path('/content/drive/MyDrive/ssl/train_images')
EVAL_DIR = Path('/content/drive/MyDrive/ssl/eval_images')
MARK_INPUT_DIR = TRAIN_DIR

CHECKPOINT_DIR = REPO_DIR / 'checkpoints'
OUTPUT_DIR = REPO_DIR / 'outputs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WHITENING_PATH = CHECKPOINT_DIR / 'whitening.pt'
ZERO_KEY_PATH = CHECKPOINT_DIR / 'key_zero.npy'
MULTI_KEY_PATH = CHECKPOINT_DIR / 'key_multi.npy'
ZERO_MARK_DIR = OUTPUT_DIR / 'zero_bit_marked'
MULTI_MARK_DIR = OUTPUT_DIR / 'multi_bit_marked'

print('TRAIN_DIR =', TRAIN_DIR)
print('EVAL_DIR =', EVAL_DIR)
print('MARK_INPUT_DIR =', MARK_INPUT_DIR)
print('WHITENING_PATH =', WHITENING_PATH)
print('ZERO_KEY_PATH =', ZERO_KEY_PATH)
print('MULTI_KEY_PATH =', MULTI_KEY_PATH)
print('ZERO_MARK_DIR =', ZERO_MARK_DIR)
print('MULTI_MARK_DIR =', MULTI_MARK_DIR)


## 5) Fit PCA-whitening

Use a folder of natural images. This step is required before marking or detection if you do not already have `checkpoints/whitening.pt`.

In [ ]:
!python -c "from src.keys import save_zerobit_key; save_zerobit_key(r'{ZERO_KEY_PATH}', d=2048)"
!python -c "from src.keys import save_multibit_key; save_multibit_key(r'{MULTI_KEY_PATH}', k=30, d=2048)"


## 7) Batch watermark a folder

This runs watermarking over every image in `MARK_INPUT_DIR` and saves the marked outputs to a folder. Change `MODE` to `zero_bit` or `multi_bit`.


In [ ]:
from pathlib import Path
import json

import torch
from tqdm import tqdm
import torchvision.transforms.functional as TF
import yaml

from src.backbone import Backbone, PCAWhitening
from src.embed import EmbedConfig, embed_multi_bit, embed_zero_bit
from src.io_utils import list_images, load_image, save_image
from src.keys import load_key

MODE = 'zero_bit'  # change to 'multi_bit' for the multi-bit workflow
MAX_IMAGES = 50
IMAGE_SIZE = None

cfg_path = REPO_DIR / 'configs' / f'{MODE}.yaml'
with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

whitening = PCAWhitening.load(WHITENING_PATH)
backbone = Backbone(whitening=whitening, feat_dim=cfg['feat_dim']).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
device = next(backbone.parameters()).device

embed_cfg = EmbedConfig(
    target_psnr=cfg['target_psnr'],
    n_iter=cfg['n_iter'],
    lr=cfg['lr'],
    lambda_w=cfg['lambda_w'],
)

input_paths = list_images(MARK_INPUT_DIR)[:MAX_IMAGES]
if not input_paths:
    raise SystemExit(f'no images found in {MARK_INPUT_DIR}')

if MODE == 'zero_bit':
    key = load_key(ZERO_KEY_PATH).to(device)
    ZERO_MARK_DIR.mkdir(parents=True, exist_ok=True)
    for path in tqdm(input_paths, desc='mark zero-bit'):
        image = load_image(path)
        if IMAGE_SIZE is not None:
            image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        marked = embed_zero_bit(backbone, image.unsqueeze(0).to(device), key=key, fpr=cfg['fpr'], config=embed_cfg)
        save_image(marked, ZERO_MARK_DIR / path.name)
    print(f'saved {len(input_paths)} zero-bit images to {ZERO_MARK_DIR}')
else:
    key = load_key(MULTI_KEY_PATH).to(device)
    MULTI_MARK_DIR.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(0)
    message_manifest = {}
    for path in tqdm(input_paths, desc='mark multi-bit'):
        image = load_image(path)
        if IMAGE_SIZE is not None:
            image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        message = torch.randint(0, 2, (cfg['n_bits'],), device=device).float() * 2.0 - 1.0
        marked = embed_multi_bit(
            backbone,
            image.unsqueeze(0).to(device),
            carriers=key,
            messages=message.unsqueeze(0),
            margin=cfg['margin'],
            config=embed_cfg,
        )
        save_image(marked, MULTI_MARK_DIR / path.name)
        message_manifest[path.name] = message.cpu().tolist()
    with open(MULTI_MARK_DIR / 'messages.json', 'w') as f:
        json.dump(message_manifest, f, indent=2)
    print(f'saved {len(input_paths)} multi-bit images to {MULTI_MARK_DIR}')


## 8) Check a marked folder

This prints detection statistics over the marked outputs instead of a single image.


In [ ]:
import json

marked_dir = ZERO_MARK_DIR if MODE == 'zero_bit' else MULTI_MARK_DIR
marked_paths = list_images(marked_dir)
if not marked_paths:
    raise SystemExit(f'no marked images found in {marked_dir}')

if MODE == 'zero_bit':
    key = load_key(ZERO_KEY_PATH).to(device)
    hits = []
    scores = []
    for path in tqdm(marked_paths, desc='detect zero-bit'):
        image = load_image(path).unsqueeze(0).to(device)
        detected, score = detect_zero_bit(backbone, image, key=key, fpr=cfg['fpr'])
        hits.append(int(detected.item()))
        scores.append(float(score.item()))
    print('TPR:', sum(hits) / max(1, len(hits)))
    print('mean score:', sum(scores) / max(1, len(scores)))
else:
    key = load_key(MULTI_KEY_PATH).to(device)
    with open(marked_dir / 'messages.json', 'r') as f:
        message_manifest = json.load(f)
    bers = []
    wers = []
    for path in tqdm(marked_paths, desc='decode multi-bit'):
        if path.name not in message_manifest:
            continue
        image = load_image(path).unsqueeze(0).to(device)
        decoded = decode_multi_bit(backbone, image, carriers=key)
        target = torch.tensor(message_manifest[path.name], device=device).float().unsqueeze(0)
        bers.append(bit_error_rate(decoded, target))
        wers.append(word_error_rate(decoded, target))
    print('decoded', len(bers), 'images')
    print('mean BER:', sum(bers) / max(1, len(bers)))
    print('mean WER:', sum(wers) / max(1, len(wers)))


## 9) Evaluate a folder

This runs the attack suite over a folder of images and reports TPR for zero-bit or BER/WER for multi-bit.


In [ ]:
!python scripts/evaluate.py --mode zero_bit --images-dir "{EVAL_DIR}" --key "{ZERO_KEY_PATH}" --whitening "{WHITENING_PATH}" --max-images 50
!python scripts/evaluate.py --mode multi_bit --images-dir "{EVAL_DIR}" --key "{MULTI_KEY_PATH}" --whitening "{WHITENING_PATH}" --max-images 50
